# Machine Learning-Based Defect Classification Using Simulated Metrology Data
### Centre for Nano Science and Engineering (CeNSE, IISc) Characterization Framework

This interactive notebook demonstrates:
1. Physics-informed simulation of multi-modal semiconductor metrology data (Ellipsometry, AFM, 4-Point Probe, C-V, CD-SEM).
2. Exploratory Data Analysis (EDA) and 300 mm wafer map defect pattern visualization.
3. Feature engineering and robust scaling.
4. Training an ensemble of ML classifiers (Random Forest, XGBoost, MLP, SVM, Logistic Regression).
5. Performance evaluation, confusion matrices, ROC-AUC, and feature importance attribution.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.physics_simulation import SemiconductorMetrologySimulator, DEFECT_CLASSES
from src.preprocessing import prepare_dataset_splits
from src.models import MetrologyClassifierTrainer
from src.evaluation import benchmark_all_models
from src.visualization import (
    plot_wafer_map,
    plot_confusion_matrix,
    plot_feature_importance,
    plot_multiclass_roc,
    plot_metrology_distribution_matrix
)

## 1. Simulate Metrology Dataset

In [ ]:
sim = SemiconductorMetrologySimulator(random_state=42)
df = sim.generate_full_dataset(total_samples=3000, balanced=False)
print(f"Generated dataset with {df.shape[0]} dies and {df.shape[1]} columns.")
df['defect_class_name'].value_counts()

## 2. Visualize 300 mm Wafer Inspection Map

In [ ]:
plot_wafer_map(df, save_path='../outputs/figures/wafer_defect_map.png')
# Display in notebook
from IPython.display import Image
Image(filename='../outputs/figures/wafer_defect_map.png')

## 3. Preprocessing, Feature Engineering & Stratified Splitting

In [ ]:
splits = prepare_dataset_splits(df, test_size=0.20, val_size=0.15, random_state=42)
print("Training feature matrix shape:", splits['X_train'].shape)
print("Test feature matrix shape:", splits['X_test'].shape)
print("Engineered features:", splits['feature_names'])

## 4. Train Model Zoo & Evaluate Benchmark Leaderboard

In [ ]:
trainer = MetrologyClassifierTrainer(random_state=42)
trainer.train_all(splits['X_train'], splits['y_train'], splits['X_val'], splits['y_val'])

leaderboard, detailed = benchmark_all_models(trainer.trained_models, splits['X_test'], splits['y_test'])
leaderboard

## 5. Model Explainability & Metrology Feature Importances

In [ ]:
rf = trainer.trained_models['RandomForest']
plot_feature_importance(
    splits['feature_names'],
    rf.feature_importances_,
    save_path='../outputs/figures/feature_importance.png',
    top_n=15
)
Image(filename='../outputs/figures/feature_importance.png')